In [74]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter      # ganti sesuai provider                     # atau langchain_qdrant, langchain_community.vectorstores.FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [75]:
import pandas as pd
data = pd.read_json('datacorpus.json')
data

,id,pasal,bab,judul,context,ayat,buku,bagian,paragraf
0,kuhp_pasal_1,1,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 1\n(1) Tidak ada satu perbuatan pun yang...,"[{'nomor': '1', 'isi': '(1) Tidak ada satu per...",NaN,NaN,NaN
1,kuhp_pasal_2,2,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 2\n(1) Ketentuan sebagaimana dimaksud da...,"[{'nomor': '1', 'isi': '(1) Ketentuan sebagaim...",NaN,NaN,NaN
2,kuhp_pasal_3,3,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 3\n(1) Dalam hal terdapat perubahan pera...,"[{'nomor': '1', 'isi': '(1) Dalam hal terdapat...",NaN,NaN,NaN
3,kuhp_pasal_4,4,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 4\nKetentuan pidana dalam Undang-Undang ...,[],NaN,NaN,NaN
4,kuhp_pasal_5,5,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 5\nKetentuan pidana dalam Undang-Undang ...,[],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
616,kuhp_pasal_617,617,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 617\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
617,kuhp_pasal_618,618,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 618\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
618,kuhp_pasal_619,619,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 619\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
619,kuhp_pasal_620,620,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 620\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN


In [76]:
data = data.drop(columns=['id','pasal','bab','judul','ayat','buku','bagian','paragraf'])
data

,context
0,Pasal 1\n(1) Tidak ada satu perbuatan pun yang...
1,Pasal 2\n(1) Ketentuan sebagaimana dimaksud da...
2,Pasal 3\n(1) Dalam hal terdapat perubahan pera...
3,Pasal 4\nKetentuan pidana dalam Undang-Undang ...
4,Pasal 5\nKetentuan pidana dalam Undang-Undang ...
...,...
616,Pasal 617\nPada saat Undang-Undang ini mulai b...
617,Pasal 618\nPada saat Undang-Undang ini mulai b...
618,Pasal 619\nPada saat Undang-Undang ini mulai b...
619,Pasal 620\nPada saat Undang-Undang ini mulai b...


In [77]:
data['context'] = (data['context']
        .str.replace("\n", " ")
        .str.replace(r" +", " ", regex=True)
        .str.strip())
data

,context
0,Pasal 1 (1) Tidak ada satu perbuatan pun yang ...
1,Pasal 2 (1) Ketentuan sebagaimana dimaksud dal...
2,Pasal 3 (1) Dalam hal terdapat perubahan perat...
3,Pasal 4 Ketentuan pidana dalam Undang-Undang b...
4,Pasal 5 Ketentuan pidana dalam Undang-Undang b...
...,...
616,Pasal 617 Pada saat Undang-Undang ini mulai be...
617,Pasal 618 Pada saat Undang-Undang ini mulai be...
618,Pasal 619 Pada saat Undang-Undang ini mulai be...
619,Pasal 620 Pada saat Undang-Undang ini mulai be...


In [78]:
data['context']

0      Pasal 1 (1) Tidak ada satu perbuatan pun yang ...
1      Pasal 2 (1) Ketentuan sebagaimana dimaksud dal...
2      Pasal 3 (1) Dalam hal terdapat perubahan perat...
3      Pasal 4 Ketentuan pidana dalam Undang-Undang b...
4      Pasal 5 Ketentuan pidana dalam Undang-Undang b...
                             ...                        
616    Pasal 617 Pada saat Undang-Undang ini mulai be...
617    Pasal 618 Pada saat Undang-Undang ini mulai be...
618    Pasal 619 Pada saat Undang-Undang ini mulai be...
619    Pasal 620 Pada saat Undang-Undang ini mulai be...
620    Pasal 621 Peraturan pelaksanaan dari Undang-Un...
Name: context, Length: 621, dtype: object

In [79]:
# Chunking — hasilkan list of Document, bukan plain string
splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=50
)

documents = []

for pasal in data['context']:
    if len(pasal) > 4000:
        # split_text → list[str], bungkus jadi Document
        splits = splitter.split_text(pasal)
        documents.extend([Document(page_content=s) for s in splits])
    else:
        documents.append(Document(page_content=pasal))


In [80]:
# Chunking — hasilkan list of Document, bukan plain string
splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=50
)

documents = []

for pasal in data['context']:
    if len(pasal) > 4000:
        # split_text → list[str], bungkus jadi Document
        splits = splitter.split_text(pasal)
        documents.extend([Document(page_content=s) for s in splits])
    else:
        documents.append(Document(page_content=pasal))


In [94]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os
# Load embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="/Users/muhammadzuamaalamin/Documents/fintunellm/model/multilingual-e5-small",
    model_kwargs={"device": "cpu"}
)

db_path = "faiss_indexe5"

# %%
# Buat atau load FAISS index
if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(documents, embeddings)  # ✅ Document objects
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


In [82]:
# ============================================================
# PENDEKATAN 1: Dense Retrieval (FAISS) — sudah kamu punya
# ============================================================
retriever_dense = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

results = retriever_dense.invoke("Berapa lama batas waktu penuntutan untuk kejahatan ringan seperti pencurian kecil?")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content[:200]}\n")

[1] Pasal 68 (1) Pidana penjara dijatuhkan untuk seumur hidup atau untuk waktu tertentu. (2) Pidana penjara untuk waktu tertentu dijatuhkan paling lama 15 (lima belas) tahun berturut turut atau paling sin

[2] Pasal 477 (1) Dipidana dengan pidana penjara paling lama 7 (tujuh) tahun atau pidana denda paling banyak kategori V, Setiap Orang yang melakukan: a. pencurian benda suci keagamaan atau kepercayaan; b.

[3] Pasal 323 (1) Setiap Orang yang melakukan perbuatan yang mengakibatkan bahaya bagi lalu lintas umum kereta api, dipidana dengan pidana penjara paling lama 7 (tujuh) tahun. (2) Jika Tindak Pidana sebag

[4] Pasal 136 (1) Kewenangan penuntutan dinyatakan gugur karena kedaluwarsa apabila: a. setelah melampaui waktu 3 (tiga) tahun untuk Tindak Pidana yang diancam dengan pidana penjara paling lama 1 (satu) t

[5] Pasal 575 (1) Setiap Orang yang secara melawan hukum merusak, menghancurkan, atau membuat tidak dapat dipakai bangunan untuk pengamanan lalu lintas udara atau menggagalkan 

In [83]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os
# Load embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="firqaaa/indo-sentence-bert-base",
    model_kwargs={"device": "cpu"}
)

db_path = "faiss_indexeindobert"

# %%
# Buat atau load FAISS index
if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(documents, embeddings)  # ✅ Document objects
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


In [87]:
retriever_dense = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

results = retriever_dense.invoke("Kalau saya menghina orang di media sosial, ancamannya berapa tahun penjara??")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content[:200]}\n")

[1] Pasal 302 (1) Setiap Orang yang Di Muka Umum menghasut dengan maksud agar seseorang menjadi tidak beragama atau berkepercayaan yang dianut di Indonesia, dipidana dengan pidana penjara paling lama 2 (d

[2] Pasal 436 Penghinaan yang tidak bersifat pencemaran atau pencemaran tertulis yang dilakukan terhadap orang lain baik Di Muka Umum dengan lisan atau tulisan, maupun di muka orang yang dihina tersebut s

[3] Pasal 451 Setiap Orang yang menahan orang dengan Kekerasan atau Ancaman Kekerasan dengan maksud untuk menempatkan orang tersebut secara melawan hukum di bawah kekuasaannya atau kekuasaan orang lain at

[4] Pasal 304 Setiap Orang yang Di Muka Umum melakukan penghinaan terhadap orang yang sedang menjalankan atau memimpin penyelenggaraan ibadah atau upacara keagamaan atau kepercayaan, dipidana dengan pidan

[5] Pasal 190 (1) Setiap Orang yang menyatakan keinginannya Di Muka Umum dengan lisan, tulisan, atau melalui media apa pun untuk meniadakan atau mengganti Pancasila sebagai das

In [93]:
retriever_dense = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

results = retriever_dense.invoke("Kalau menghasut orang untuk membenci agama tertentu di media sosial, hukumannya apa??")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content[:200]}\n")

[1] Pasal 302 (1) Setiap Orang yang Di Muka Umum menghasut dengan maksud agar seseorang menjadi tidak beragama atau berkepercayaan yang dianut di Indonesia, dipidana dengan pidana penjara paling lama 2 (d

[2] Pasal 300 Setiap Orang Di Muka Umum yang: a. melakukan perbuatan yang bersifat permusuhan; b. menyatakan kebencian atau permusuhan; atau c. menghasut untuk melakukan Kekerasan, atau diskriminasi, terh

[3] Pasal 304 Setiap Orang yang Di Muka Umum melakukan penghinaan terhadap orang yang sedang menjalankan atau memimpin penyelenggaraan ibadah atau upacara keagamaan atau kepercayaan, dipidana dengan pidan

[4] Pasal 436 Penghinaan yang tidak bersifat pencemaran atau pencemaran tertulis yang dilakukan terhadap orang lain baik Di Muka Umum dengan lisan atau tulisan, maupun di muka orang yang dihina tersebut s

[5] Pasal 530 Setiap Pejabat atau orang lain yang bertindak dalam suatu kapasitas Pejabat resmi, atau orang yang bertindak karena digerakkan atau sepengetahuan Pejabat publik m